In [1]:
import pandas as pd
import re
import json


input_path = "jty2_.txt"  

with open(input_path, "r", encoding="utf-8") as f:
    text = f.read()

pattern = r"(\/[^\n]+\/(normal|abnormal)\/[^\n]+)\s*[\s\S]*?(?:```json\s*([\s\S]*?)\s*```|(\{[\s\S]*?\}))"
matches = re.findall(pattern, text, re.DOTALL)

rows = []

for match in matches:
    
    path, truth, json_block, json_raw = match
    json_str = json_block if json_block else json_raw
    
    try:
        data = json.loads(json_str)

        is_scam = bool(data.get("is_scam"))

        if (is_scam and truth == "abnormal") or (not is_scam and truth == "normal"):
            result = True
        else:
            result = False

        rows.append({
            "filename": path.split("/")[-1].strip(),
            "truth": truth,
            "is_scam": data.get("is_scam"),
            "result": result,
            "is_correct": data.get("is_scam"),
            "confidence": data.get("confidence"),
            "risk": data.get("risk"),
            "evidence": "; ".join(data.get("evidence", [])),
            "explanation": data.get("explanation"),            
        })
    except json.JSONDecodeError as e:
        print(f"JSON 파싱 오류 발생 ({path}): {e}")

df = pd.DataFrame(rows, columns=["filename", "truth", "is_scam", "result", "confidence", "risk", "evidence", "explanation"])

output_path = "scam_results.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"CSV 파일 생성 완료: {output_path}")
print(df)


JSON 파싱 오류 발생 (/home/ubuntu/cybercop/video_20251104/normal/0016.mp4): Invalid control character at: line 1 column 3 (char 2)
JSON 파싱 오류 발생 (/home/ubuntu/cybercop/video_20251104/normal/0014.mp4): Invalid control character at: line 1 column 3 (char 2)
JSON 파싱 오류 발생 (/home/ubuntu/cybercop/video_20251104/normal/0010.mp4): Invalid control character at: line 1 column 3 (char 2)
JSON 파싱 오류 발생 (/home/ubuntu/cybercop/video_20251104/abnormal/0010.mp4): Extra data: line 1 column 10 (char 9)
CSV 파일 생성 완료: scam_results.csv
    filename     truth  is_scam  result  confidence  risk  \
0   0025.mp4    normal     True   False        0.95  high   
1   0011.mp4    normal    False    True        0.55   low   
2   0024.mp4    normal    False    True        0.56   low   
3   0001.mp4    normal    False    True        0.35   low   
4   0006.mp4    normal    False    True        0.50   low   
5   0005.mp4    normal    False    True        0.50   low   
6   0023.mp4    normal    False    True        0.55   low

In [3]:
df.groupby(["truth", "result"]).agg({"filename":"count"})

filename
truth    result          
abnormal False         14
         True           5
normal   False          1
         True          20

In [4]:
df[df['result']==True]

,filename,truth,is_scam,result,confidence,risk,evidence,explanation
1,0011.mp4,normal,False,True,0.55,low,알바하지Street scene scenes; 아이폰; 구매; 영수증,The video does not contain any clear or strong...
2,0024.mp4,normal,False,True,0.56,low,"Bojung's must-do; Bojung's must-know; No pain,...",The video appears to be an informative guide a...
3,0001.mp4,normal,False,True,0.35,low,The video contains discussions about part-time...,The content focuses on labor issues and discus...
4,0006.mp4,normal,False,True,0.50,low,"Korean text appears on the screen, discussing ...",The video contains Korean discussions with no ...
5,0005.mp4,normal,False,True,0.50,low,Various foods showcased; Prices displayed in K...,The video appears to show typical food offerin...
6,0023.mp4,normal,False,True,0.55,low,"The video showcases a person cooking, emphasiz...",The focus is on demonstrating the features and...
7,0007.mp4,normal,False,True,0.65,low,"Collecting money for a project, then leaving; ...",The video primarily shows routine job site ope...
8,0021.mp4,normal,False,True,0.75,low,ì£© í»ìŠ° í”¿ê·° í£¼ ìë¡¨ê³ êµ­ í»ìš°ê² ...,The video appears to be an informational adver...
9,0027.mp4,normal,False,True,0.53,low,The video shows a person eating various meals ...,The video appears to focus on someone's experi...
10,0012.mp4,normal,False,True,0.75,low,alent_인가,The video appears to be a light-hearted and hu...


In [5]:
df[df['result']==False]

,filename,truth,is_scam,result,confidence,risk,evidence,explanation
0,0025.mp4,normal,True,False,0.95,high,"Install and run a scam program, pretending to ...",The video shows deceptive tactics commonly use...
21,0025.mp4,abnormal,False,False,0.50,low,"The person explains something, pointing to the...",The video shows a casual explanation of what a...
22,0011.mp4,abnormal,False,False,0.35,low,The video features a character with a balaclav...,While there are hints suggesting a scam scenar...
23,0034.mp4,abnormal,False,False,0.65,low,JP摩根城堡已对B币的比尔米莱投入150亿韩元; JP摩根城堡在投资前提到该股票有可能翻倍;...,视频显示了关于SKHainix股票的信息，JP摩根城堡被提及为已经大量投资于该股票。虽然这些...
25,0033.mp4,abnormal,False,False,0.35,low,,"The content is primarily informative, discussi..."
26,0004.mp4,abnormal,False,False,0.35,low,10월 전에는国 사라; 嚼嚼을만 남아Megacore,The video discusses investment tips and strate...
28,0016.mp4,abnormal,False,False,0.56,low,No clear indicators of financial gain or inves...,The content appears focused on providing some ...
29,0005.mp4,abnormal,False,False,0.55,low,ガサング会觉得구매대행 모집 한 달이면 마음에 드는 차를 샀습니다。; 十字路口的交通,The video appears to be informative or educati...
30,0018.mp4,abnormal,False,False,0.75,low,A man in a white uniform demonstrating martial...,The video shows an instructional session where...
31,0014.mp4,abnormal,False,False,0.35,low,You can easily recognize a scam.; You will reg...,The video appears to be an informational or wa...
